# Sakura: Colab GPU と PC で Qwen を使う

## 目的

まずは **Qwen3.5-0.8B 全体を Colab GPU に載せ、キャッシュを有効にして文章を生成**します。PC 接続を使う場合も、モデル計算は GPU 側にまとめます。PC は受付・クライアントを担当します。

| モード | 実行場所 | Colab の対象 |
|---|---|---|
| `local`（初期設定） | ノートブック内で単体推論 | 無料枠でも利用可能。GPU の提供は保証されません |
| `gateway_worker` | PC が受付、Colab が仕事を取得して GPU 推論 | 正の compute unit 残高を持つ対象の有料プラン、Enterprise、または自己管理の計算環境 |

無料の managed runtime で compute unit 残高がない場合、分散 worker や SSH、ノートブック UI の迂回は禁止されています。managed runtime 全般で remote proxy 接続も禁止されています。このノートブックはトンネルや Colab ポート公開を作りません。[Colab 公式 FAQ](https://research.google.com/colaboratory/faq.html#limitations-and-restrictions)

モデル revision: [`2fc06364715b967f1860aea9cf38778875588b17`](https://huggingface.co/Qwen/Qwen3.5-0.8B/tree/2fc06364715b967f1860aea9cf38778875588b17)。これは dense の小型モデルです。35B MoE の学習・分割実装とは別の入口です。

## セットアップ

Colab の **ランタイム → ランタイムのタイプを変更 → GPU** を選び、次のセルから順に実行します。

最初は `MODE="local"` のままにしてください。PC 接続を使うときだけ `MODE="gateway_worker"` とし、利用環境が条件を満たすことを確認してフラグを変更します。PC 接続の追加手順は後半にあります。

In [ ]:
REPO_URL = "https://github.com/Summer110622/agora.git"
REPO_REF = "codex/sakura-community-inference"
REPO_DIRECTORY = "/content/agora"

MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_REVISION = "2fc06364715b967f1860aea9cf38778875588b17"
DEVICE = "cuda"  # GPU がないときに CPU を選ぶ場合だけ "cpu" に変更
PROMPT = "Pythonで2つの数を足す関数を、短い説明と一緒に書いてください。"
MAX_NEW_TOKENS = 64
MODE = "local"  # "local" または "gateway_worker"

# 対象の有料プランで正の compute unit 残高がある、Enterprise、または自己管理の
# 計算環境の場合だけ True にします。コードは契約・残高を自動判定できません。
ELIGIBLE_DISTRIBUTED_RUNTIME = False

### 1. 推論用パッケージを入れる

Colab が提供する CUDA 対応 PyTorch を維持し、推論用の依存だけを固定します。これは学習用 `torch==2.10.0` の環境とは別です。Python 3.11 / 3.12 が対象です。ローカルの実装試験は CPU / PyTorch 2.10 で行っており、Colab 提供の GPU / PyTorch の組合せは実機で確認してください。

ブランチが更新されても、取得した実際の commit を表示するので実行元を記録できます。既存の checkout が別ブランチの場合は処理を止めます。インストール後に既存の import が残る場合は、ランタイムを再起動して最初から実行してください。

In [ ]:
import importlib.metadata
from pathlib import Path
import subprocess
import sys

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("このノートブックは Python 3.11 / 3.12 が対象です。")
if MODE not in {"local", "gateway_worker"}:
    raise ValueError("MODE は local / gateway_worker を選んでください。")
if MODE == "gateway_worker" and not ELIGIBLE_DISTRIBUTED_RUNTIME:
    raise RuntimeError("PC 接続は対象の有料残高あり / Enterprise / 自己管理環境で実行してください。無料枠は MODE='local' を使います。")

repo = Path(REPO_DIRECTORY)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo)], check=True)
else:
    branch = subprocess.check_output(["git", "-C", str(repo), "branch", "--show-current"], text=True).strip()
    if branch != REPO_REF:
        raise RuntimeError("既存 checkout のブランチが異なります。REPO_DIRECTORY を新しい場所に変更してください。")
if not (repo / "sakura_inference.py").is_file():
    raise RuntimeError("この ref に推論コードがありません。PR のブランチを指定してください。")

torch_version_before = importlib.metadata.version("torch")
inference_dependencies = [
    "transformers==5.9.0", "huggingface-hub==1.5.0", "safetensors==0.7.0",
    "cryptography==50.0.2", "psutil==7.2.2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *inference_dependencies], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(repo / "sakura")], check=True)
assert importlib.metadata.version("torch") == torch_version_before
print("取得 commit:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip())

### 2. 実際の GPU を確認する

`cuda` を選んだのに GPU が使えない場合は、CPU に自動変更せず停止します。GPU に空きがない場合は Colab の設定を確認するか、`DEVICE="cpu"` を明示して単体推論を試してください。

In [ ]:
import torch

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU がありません。Colab の GPU 設定を確認するか DEVICE='cpu' を選んでください。")
print({"torch": torch.__version__, "cuda_runtime": torch.version.cuda, "device": DEVICE})
if DEVICE == "cuda":
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print({"gpu": torch.cuda.get_device_name(0), "free_GiB": round(free_bytes / 2**30, 2),
           "total_GiB": round(total_bytes / 2**30, 2)})

## 手順: ノートブック内で単体推論

### 3. モデルを一度だけ読み込む

初回は固定 revision のモデルを Hugging Face からダウンロードします。公開モデルなのでトークンの入力は不要です。`dtype="auto"` は GPU に合う計算型を選び、`max_context=4096` で入力と生成の長さを制限します。

35B を使う場合は全 expert の保存容量・VRAM と cache が必要です。A3B という表記だけで小型 GPU に入るとは見積もれません。

In [ ]:
if MODE == "local":
    from sakura.inference_generate import HFGenerator
    generator = HFGenerator.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, device=DEVICE, dtype="auto", max_context=4096,
    )
else:
    print("PC worker モードなので、このセルではモデルを読み込みません。")

### 4. 生成する

このセルだけを再実行すると、読み込んだモデルを使い続けられます。生成コードは表示するだけです。生成途中の KV / recurrent cache を有効にし、通常は chat template を使用します。

速度の数値はこの実行の実測です。GPU 型、入力長、初回の準備、通信時間が違う測定はそのまま比較できません。

In [ ]:
import json

if MODE == "local":
    result = generator.generate(PROMPT, max_new_tokens=MAX_NEW_TOKENS)
    print(result["text"])
    print(json.dumps(result.get("metrics", {}), ensure_ascii=False, indent=2))

## 手順: PC から Colab GPU に依頼する（対象環境のみ）

`MODE="gateway_worker"` に切り替えて最初から実行すると、ここから PC 接続を使えます。**無料の managed runtime で compute unit 残高がない場合は使わないでください。**

通信の向きは `PC の受付 ← Colab から HTTPS で仕事を取得` です。PC のクライアントは PC 内の受付に依頼します。Colab にサーバー・着信ポート・Google 認証 Cookie を用意する必要はありません。

### 5. PC で準備する

この PR のブランチを PC に取得し、推論環境を用意します。詳しい導入は [INFERENCE.md](https://github.com/Summer110622/agora/blob/codex/sakura-community-inference/sakura/INFERENCE.md) を参照してください。次は **PC のリポジトリ直下**で実行します。

```powershell
python sakura_inference.py init --directory sakura/runs/inference --model Qwen/Qwen3.5-0.8B --revision 2fc06364715b967f1860aea9cf38778875588b17
python sakura_inference.py gateway --manifest sakura/runs/inference/inference-run.json --identity sakura/runs/inference/pc.private.json --host 0.0.0.0 --allow-external --port 49201 --tls-cert C:/certs/fullchain.pem --tls-key C:/certs/privkey.pem
```

`C:/certs/fullchain.pem` と `C:/certs/privkey.pem` は利用者の実際の証明書に置き換えます。**Colab から到達できる PC の hostname と、有効な信頼済み証明書、到達可能な PC 側ポート**が必要です。例は `https://your-pc.example:49201/rpc` です。自己管理 VPC の既存経路でも利用できます。

既存の HTTPS 終端を使う場合は、gateway を `--host 127.0.0.1 --port 49201` で HTTP 起動し、その入口から転送できます。native TLS の gateway に `http://127.0.0.1` で接続することはできません。

インターネット越しに PC が NAT の内側にあるだけでは Colab から接続できません。HTTPS / private routing の設定は利用者の環境ごとに必要です。Colab 側に公開ポートを作る方法でこれを迂回しません。入口がない場合も前半の単体推論は利用できます。

作成した **`inference-run.json` と `colab.private.json` の 2 ファイルだけ**を次のセルでアップロードします。`pc.private.json` は PC に保管してください。

### 6. PC の HTTPS URL を入力する

URL にパスワード・token を含めないでください。通信は Ed25519 の署名で相手と manifest を確認し、HTTPS が通信内容を保護します。証明書エラーを無視する設定は使いません。

In [ ]:
if MODE == "gateway_worker":
    from urllib.parse import urlsplit
    pc_endpoint = input("PC の HTTPS RPC URL: ").strip()
    parsed = urlsplit(pc_endpoint)
    if (parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password
            or parsed.query or parsed.fragment or not parsed.path.endswith("/rpc")):
        raise ValueError("認証情報・query を含まない https://.../rpc を指定してください。")

### 7. 公開 manifest と Colab 用の鍵だけをアップロードする

ファイル選択で `inference-run.json` と `colab.private.json` だけを選びます。秘密鍵の中身は表示しません。ノートブックは出力を保存しない設定で配布しています。Google Drive への鍵保存や Git への追加は不要です。

In [ ]:
if MODE == "gateway_worker":
    import contextlib
    import tempfile
    from google.colab import files
    session_workspace = tempfile.TemporaryDirectory(prefix="sakura-inference-", dir="/content")
    session_directory = Path(session_workspace.name)
    with contextlib.chdir(session_directory):
        uploaded = files.upload()
    required_files = {"inference-run.json", "colab.private.json"}
    if set(uploaded) != required_files:
        uploaded.clear()
        session_workspace.cleanup()
        raise ValueError("inference-run.json と colab.private.json の 2 ファイルだけを選択してください。pc.private.json はアップロードしません。")
    for name in required_files:
        (session_directory / name).write_bytes(uploaded[name])
    (session_directory / "colab.private.json").chmod(0o600)
    uploaded.clear()
    print("必要な 2 ファイルを読み込みました。秘密鍵の内容は表示しません。")

### 8. Colab worker を動かす

このセルは依頼を待つ間、実行中のままになります。PC の HTTPS 入口に outbound 接続し、生成ごとに仕事を取得します。token ごとに PC と GPU を往復する構成ではありません。停止ボタンを押すと子プロセスも止めます。

別の **PC ターミナル**から次を実行します。

```powershell
python sakura_inference.py chat --manifest sakura/runs/inference/inference-run.json --identity sakura/runs/inference/pc.private.json --endpoint https://your-pc.example:49201/rpc --prompt "Pythonで足し算の関数を書いて" --max-new-tokens 64
```

モデルの初回ダウンロード・読込が終わるまで待ってください。Colab の runtime が切断された場合は worker の再接続が必要です。PC の応答時間にはキュー待ち・通信・生成の時間が加わります。

In [ ]:
worker_process = None
if MODE == "gateway_worker":
    if not ELIGIBLE_DISTRIBUTED_RUNTIME:
        raise RuntimeError("対象の利用環境を確認してから実行してください。")
    command = [
        sys.executable, str(repo / "sakura_inference.py"), "worker",
        "--manifest", str(session_directory / "inference-run.json"),
        "--identity", str(session_directory / "colab.private.json"),
        "--endpoint", pc_endpoint, "--device", DEVICE, "--dtype", "auto",
    ]
    worker_process = subprocess.Popen(command)
    try:
        returncode = worker_process.wait()
        if returncode:
            raise RuntimeError(f"worker が終了しました (exit={returncode})。上のエラーを確認してください。")
    except KeyboardInterrupt:
        worker_process.terminate()
        try:
            worker_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            worker_process.kill()
            worker_process.wait()
        print("worker を停止しました。")
    finally:
        if worker_process.poll() is None:
            worker_process.terminate()

## 確認すること

- 単体推論: 実際の GPU 名と生成結果、実測 metrics が表示されること。
- PC 接続: HTTPS が Colab から到達でき、PC の `chat` が返ること。署名・manifest 不一致や TLS エラーは設定を直して再実行すること。
- 保存: 秘密鍵・アップロードされたファイルの中身・認証情報がセル出力に残っていないこと。共有前に出力をクリアすること。

**配布時の検証範囲:** ノートブック形式と Python セル構文、コマンド/API の整合をローカルで確認しています。実際の Colab GPU と利用者の PC の HTTPS 経路では未実行です。ここに速度の保証値や接続成功の実績は載せていません。

## 次の手順

1. 同じ入力長・生成数で `sakura_inference.py local --benchmark` を使い、cache 有効/無効の実測を比較します。Notebook のモデルと同時に二重ロードせず、別の新しい実行環境で測定してください。
2. 1 台にモデル全体が入らない場合だけ、[INFERENCE.md の stage 分割](https://github.com/Summer110622/agora/blob/codex/sakura-community-inference/sakura/INFERENCE.md#stage-分割を使う場合)を検討します。WAN 分割は token ごとに通信するため、速度が下がる場合があります。
3. 利用終了時は worker を停止し、以下の片付けセルを明示的に実行します。Colab の切断時にファイルやモデルは失われるため、PC の元の鍵を保管してください。

`google.colab.kernel.proxyPort` はブラウザの現在のユーザー/開いているノートブック用の仕組みで、PC の通常の HTTP クライアントから使える公開 API として扱いません。[公式 colabtools 実装](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py)

In [ ]:
CLEANUP_WHEN_DONE = False  # 利用を終えるときだけ True にして、このセルを実行
if CLEANUP_WHEN_DONE:
    if worker_process is not None and worker_process.poll() is None:
        worker_process.terminate()
        try:
            worker_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            worker_process.kill()
            worker_process.wait()
    if MODE == "gateway_worker":
        session_workspace.cleanup()
    if "generator" in globals():
        del generator
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("worker と、このセッション用にアップロードした Colab 鍵を片付けました。")